In [ ]:
import os
import numpy as np
import pandas as pd


FAULT_T = 1.0
T_START, T_END = 0.95, 3.2
RAW_DT = 0.001 
TARGET_DT = 0.05    
DECIM = int(round(TARGET_DT / RAW_DT))  # 50

EPS = 1e-12

def load_powerfactory_csv(path: str) -> pd.DataFrame:
    """
    Loads a PowerFactory CSV with:
      - first line: element names
      - second line: variable names (includes "Time in s" in first col)
    Returns a DataFrame with flattened column names.
    """
    df = pd.read_csv(path, header=[0, 1])

    # Flatten MultiIndex columns: (element, variable) -> "element__variable"
    flat_cols = []
    for a, b in df.columns:
        a = str(a).strip()
        b = str(b).strip()
        if a == "All calculations" and "Time in s" in b:
            flat_cols.append("Time in s")
        else:
            flat_cols.append(f"{a}__{b}")
    df.columns = flat_cols
    return df

def slice_and_decimate(df: pd.DataFrame) -> tuple[np.ndarray, pd.DataFrame]:
    """
    Slices to [T_START, T_END] and decimates from 1kHz to 20Hz by taking every 50th sample.
    Assumes fixed dt=0.001. Uses time-based masking for robustness.
    """
    t = pd.to_numeric(df["Time in s"], errors="coerce")
    df = df.loc[t.notna()].copy()
    t = t.loc[t.notna()].to_numpy()

    mask = (t >= T_START) & (t <= T_END)
    df_w = df.loc[mask].reset_index(drop=True)

    # Decimate by simple indexing (fast)
    df_w = df_w.iloc[::DECIM].reset_index(drop=True)

    t_ds = df_w["Time in s"].to_numpy()
    X_ds = df_w.drop(columns=["Time in s"])

    # Make sure numeric
    for c in X_ds.columns:
        X_ds[c] = pd.to_numeric(X_ds[c], errors="coerce")
    X_ds = X_ds.fillna(method="ffill").fillna(method="bfill").fillna(0.0)

    return t_ds, X_ds

def channel_summary_features(t: np.ndarray, x: np.ndarray) -> dict:
    """
    Computes window-level summary features for one channel.
    Uses pre-fault segment [T_START, FAULT_T) as baseline.
    """
    pre_mask = (t >= T_START) & (t < FAULT_T)
    x_pre = x[pre_mask] if np.any(pre_mask) else x[:max(1, len(x)//10)]

    pre_mean = float(np.mean(x_pre))
    pre_std  = float(np.std(x_pre)) + EPS

    dev = x - pre_mean
    abs_dev = np.abs(dev)

    # Derivative (t is uniform after decimation, but compute generally)
    if len(x) >= 2:
        dx = np.diff(x) / np.diff(t)
        max_abs_dx = float(np.max(np.abs(dx)))
    else:
        max_abs_dx = 0.0

    idx_peak = int(np.argmax(abs_dev)) if len(x) else 0
    t_at_peak = float(t[idx_peak] - FAULT_T) if len(x) else 0.0

    feats = {
        "pre_mean": pre_mean,
        "pre_std": pre_std,
        "min": float(np.min(x)),
        "max": float(np.max(x)),
        "ptp": float(np.max(x) - np.min(x)),
        "max_abs_dev": float(np.max(abs_dev)),
        "rms_dev": float(np.sqrt(np.mean(dev**2))),
        "auc_abs_dev": float(np.sum(abs_dev) * (t[1] - t[0]) if len(t) >= 2 else 0.0),
        "max_abs_dx": max_abs_dx,
        "t_at_max_abs_dev": t_at_peak,
        "final_minus_prefault": float(x[-1] - pre_mean) if len(x) else 0.0,
    }
    return feats

def featurise_simulation(csv_path: str) -> pd.DataFrame:
    """
    Produces a 1-row DataFrame: engineered features for this simulation.
    """
    df_raw = load_powerfactory_csv(csv_path)
    t, X = slice_and_decimate(df_raw)

    row = {}
    for col in X.columns:
        feats = channel_summary_features(t, X[col].to_numpy())
        for k, v in feats.items():
            row[f"{col}__{k}"] = v

    # optionally store metadata (useful for debugging / grouping)
    row["meta__file"] = os.path.basename(csv_path)
    row["meta__t_start"] = T_START
    row["meta__t_end"] = T_END
    row["meta__dt"] = TARGET_DT

    return pd.DataFrame([row])

if __name__ == "__main__":
    in_file = r"C:\Users\M Hasan Rosyid\Desktop\Dataset_Project\LHS\LHS\in1\lhs_results/0_Line 02 - 25_load=0.9784_wind=0.365_0.8582_0.3085.csv"
    out_file = r"C:\Users\M Hasan Rosyid\Desktop\PowerSystemDataset\tabulardata\new.csv"

    features_df = featurise_simulation(in_file)
    features_df.to_csv(out_file, index=False)

    print("Saved:", out_file)
    print("Shape:", features_df.shape)


Saved: C:\Users\M Hasan Rosyid\Desktop\PowerSystemDataset\tabulardata\new.csv
Shape: (1, 2765)


C:\Users\M Hasan Rosyid\AppData\Local\Temp\ipykernel_7932\1311109430.py:56: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  X_ds = X_ds.fillna(method="ffill").fillna(method="bfill").fillna(0.0)
